# Lesson 4: Automated Testing & Load Testing
### VisionStream — "No Code Without Tests"

---

## Learning Objectives

By the end of this lesson, you will be able to:

1. **Write** pytest unit tests using fixtures and the Given/When/Then pattern
2. **Design** edge-case test scenarios for input validation
3. **Measure** test coverage and interpret what it means
4. **Run** a load test with Locust and read the QPS/latency charts
5. **Identify** system bottlenecks from load test results

## Section 1: Testing Philosophy

### Why Tests Are Non-Negotiable in Industry

In every software company, code without tests is considered **unfinished**.
Here is why:

| Without tests | With tests |
|---------------|------------|
| Bugs discovered by users | Bugs caught before deployment |
| Afraid to refactor (might break things) | Confident to refactor (tests catch regressions) |
| "It worked on my machine" | Consistent behavior across all environments |
| Manual testing every release | Automated verification in < 30 seconds |

### Types of Tests (relevant for VisionStream)

| Type | What it tests | Speed | Dependencies |
|------|--------------|-------|-------------|
| **Unit test** | One function in isolation | Milliseconds | Mocked DB, no network |
| **Integration test** | Multiple components together | Seconds | Real PostgreSQL |
| **Load test** | System under high concurrency | Minutes | Running server |

In Lesson 4, you will write **unit tests** (fast, isolated) and run a **load test**.

### The Testing Pyramid

```
         ▲
        /E2E\          ← End-to-end tests: few, slow, expensive
       /─────\         ← Integration tests: moderate
      /Unit   \        ← Unit tests: many, fast, cheap
     /─────────\
```

Build most of your test coverage with unit tests.

## Section 2: pytest Fundamentals

### Test Discovery

pytest automatically finds and runs tests if:
- Files are named `test_*.py` or `*_test.py`
- Functions are named `test_*`
- Classes are named `Test*`

```bash
# Run all tests:
pytest tests/ -v

# Run one file:
pytest tests/test_devices.py -v

# Run one test by name:
pytest tests/test_devices.py::TestDeviceRegistration::test_register_device_success -v

# Run with coverage report:
pytest tests/ -v --cov=app --cov-report=term-missing
```

### Fixtures

A **fixture** is a reusable setup function. Use `@pytest.fixture` to define one.
pytest automatically injects fixtures as function parameters.

```
conftest.py                     test_devices.py
─────────────────────────       ──────────────────────────────────
@pytest.fixture                 def test_register_success(client):
def client():                       # 'client' is automatically
    ...                             # injected by pytest
    yield TestClient(app)           response = client.post(...)
    ...cleanup...                   assert response.status_code == 201
```

### The Given/When/Then Pattern

Every test has three parts:

```python
def test_register_device_duplicate_returns_409(client):
    # GIVEN: A device is already registered
    payload = {"device_id": "dup-001", "owner_name": "Alice"}
    client.post("/devices/register", json=payload)

    # WHEN: The same device_id is registered again
    response = client.post("/devices/register", json=payload)

    # THEN: Response is 409 Conflict
    assert response.status_code == 409
```

## Section 3: Illustrative Code — pytest Fixtures

Study how `conftest.py` fixtures work. The key pattern here is **dependency override**:
replacing the production `get_db()` dependency with a test database session.

In [ ]:
# ── ILLUSTRATIVE EXAMPLE ONLY — Study the pattern ──

# This illustrates how FastAPI's dependency_overrides works.
# Implement the real version in tests/conftest.py

# from fastapi.testclient import TestClient
# from sqlalchemy import create_engine
# from sqlalchemy.orm import sessionmaker
# import pytest

# Step 1: Create an in-memory SQLite database for testing
# (No PostgreSQL installation needed — tests run anywhere)
#
# test_engine = create_engine(
#     "sqlite:///:memory:",
#     connect_args={"check_same_thread": False},  # Required for SQLite
# )
# TestingSessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=test_engine)

# Step 2: Create all tables in the test database
# Base.metadata.create_all(bind=test_engine)

# Step 3: Override the get_db dependency
# @pytest.fixture
# def client():
#     def override_get_db():
#         db = TestingSessionLocal()
#         try:
#             yield db
#         finally:
#             db.close()
#
#     # This line replaces get_db() for all tests using this fixture
#     app.dependency_overrides[get_db] = override_get_db
#
#     with TestClient(app) as test_client:
#         yield test_client           # <- the fixture's value
#
#     # Clean up after the test
#     app.dependency_overrides.clear()


# Key concept: Why use scope="function" vs scope="module" vs scope="session"?
scope_explanation = {
    "function": "Creates a fresh DB for EACH test — maximum isolation (use this)",
    "module": "One DB shared across all tests in one file — faster but tests can interfere",
    "session": "One DB for the entire test run — fastest but highest interference risk",
}
for scope, description in scope_explanation.items():
    print(f"scope='{scope}': {description}")

## Section 4: Edge Case Design

Writing a test for the happy path is easy. **Edge cases are where bugs hide.**

For every input field, ask:
- What is the minimum valid value? What is one below the minimum?
- What is the maximum valid value? What is one above the maximum?
- What if the field is missing entirely?
- What if the field has the wrong type?

### VisionStream Telemetry Edge Case Matrix

| Field | Valid Range | Test: too low | Test: too high | Test: missing |
|-------|-------------|---------------|----------------|---------------|
| `gps_lat` | [-90, 90] | -91.0 → 422 | 95.0 → 422 | → 422 |
| `gps_lon` | [-180, 180] | -181.0 → 422 | 200.0 → 422 | → 422 |
| `battery_level` | [0, 100] or None | -1 → 422 | 110 → 422 | None → 201 (optional) |
| `obstacle_distance_cm` | > 0 or None | -5 → 422 | N/A | None → 201 (optional) |
| `device_id` | registered | — | — | unregistered → 404 |

### Business Logic Edge Cases (harder to spot)

| Scenario | Expected behavior |
|----------|------------------|
| Duplicate device registration | 409, NOT 500 |
| Upload telemetry for unregistered device | 404, NOT 500 |
| `GET /alerts/history` with no alerts | `[]`, NOT 404 |
| `GET /telemetry/history` for new device | `[]`, NOT 404 |
| Alert created ONLY when `alert_type != "NONE"` | Verify NONE telemetry creates no alert |

## Section 5: Illustrative Code — Test Structure Pattern

Study the pattern below before implementing the real tests in `tests/test_*.py`.

In [ ]:
# ── ILLUSTRATIVE EXAMPLE ONLY — Study the pattern ──

# Pattern: Test class with multiple related test cases

# class TestExampleEndpoint:
#
#     def test_valid_request_returns_201(self, client):
#         """Happy path — valid data, expected success."""
#         payload = {"name": "valid-name", "value": 42}
#         response = client.post("/example", json=payload)
#         assert response.status_code == 201
#         data = response.json()
#         assert data["name"] == "valid-name"   # Check response body too
#
#     def test_missing_required_field_returns_422(self, client):
#         """Pydantic validation: missing required field."""
#         payload = {}  # Missing 'name' field
#         response = client.post("/example", json=payload)
#         assert response.status_code == 422
#         # Optionally check the error detail:
#         errors = response.json()["detail"]
#         assert any(e["loc"] == ["body", "name"] for e in errors)
#
#     def test_value_out_of_range_returns_422(self, client):
#         """Pydantic validation: Field constraint violation."""
#         payload = {"name": "test", "value": -999}  # Invalid: below minimum
#         response = client.post("/example", json=payload)
#         assert response.status_code == 422


# Pattern: Coverage report interpretation
#
# Run: pytest tests/ --cov=app --cov-report=term-missing
#
# Output example:
# Name                               Stmts   Miss  Cover   Missing
# app/routers/devices.py               25      5    80%   45-52
# app/services/device_service.py       18      0   100%
# app/services/alert_service.py        30     12    60%   88-102
#
# 'Miss' = lines that no test executes
# 'Missing' = line numbers not covered
# Target: > 80% overall coverage

coverage_tips = [
    "100% coverage does not mean bug-free — it means every line was executed",
    "Focus on testing behavior (what the code does), not implementation details",
    "Aim for 80%+ coverage, not 100% — some code is hard to test (startup events, etc.)",
    "Uncovered lines (Miss column) often reveal untested error paths",
]
for tip in coverage_tips:
    print(f"→ {tip}")

## Section 6: Load Testing with Locust

Unit tests verify correctness. **Load tests** verify that your system can handle real-world traffic volumes.

### Key Metrics

| Metric | Meaning | Healthy value (approximate) |
|--------|---------|-----------------------------|
| **RPS** | Requests per second processed | Higher = better |
| **P50 latency** | Median response time | < 100ms |
| **P95 latency** | 95% of requests complete within this time | < 500ms |
| **P99 latency** | 99th percentile | < 2000ms |
| **Failure rate** | % of requests that returned 4xx/5xx | < 0.1% |

### The "Knee of the Curve"

```
Response
Latency
   ▲
500ms│                              ╭────
     │                           ╭─╯
100ms│──────────────────────────╯   ← Knee: system starts struggling
     │
  10ms│──────────────────
     └─────────────────────────────►
       100    500    1000   2000     Concurrent Users
```

Find where the latency curve "bends upward" — that is your system's current capacity limit.

### Locust Architecture

```
locust -f locust/locustfile.py --host=http://localhost:8000

Web UI at http://localhost:8089:
  → Set: Number of users (e.g., 1000)
  → Set: Spawn rate (e.g., 10 users/second)
  → Click: Start swarming

Locust spawns N concurrent virtual users.
Each user runs tasks in a loop:
  - upload_telemetry (weight 10 = runs 10x more often)
  - check_alert_history (weight 1)
```

## Section 7: Illustrative Code — Locust User Pattern

Study how a `HttpUser` class is structured before implementing `locust/locustfile.py`.

In [ ]:
# ── ILLUSTRATIVE EXAMPLE ONLY — Study the pattern ──

# from locust import HttpUser, task, between
# import random
# from datetime import datetime, timezone

# class ExampleUser(HttpUser):
#     """
#     Each instance of this class is ONE virtual concurrent user.
#     Locust creates N instances (N = users you specify in Web UI).
#     """
#
#     # Each user waits 1-3 seconds between tasks (simulates real user behavior)
#     wait_time = between(1, 3)
#
#     def on_start(self):
#         """Called once when the virtual user is created."""
#         # Do setup here (e.g., login, register device)
#         pass
#
#     @task(10)  # weight=10: this task runs 10x more often than weight-1 tasks
#     def heavy_task(self):
#         """High-frequency operation (e.g., upload sensor data)"""
#         payload = {"value": random.randint(1, 100)}
#         with self.client.post("/example", json=payload, catch_response=True) as resp:
#             if resp.status_code != 201:
#                 resp.failure(f"Expected 201, got {resp.status_code}")
#
#     @task(1)  # weight=1: runs infrequently
#     def light_task(self):
#         """Low-frequency operation (e.g., check history)"""
#         self.client.get("/example/history")


# Understanding task weights:
# @task(10) and @task(1) means:
# For every 11 tasks executed:
#   - heavy_task runs 10 times (≈ 91%)
#   - light_task runs 1 time  (≈ 9%)
#
# This matches real helmet behavior:
#   - Telemetry upload: every 1 second per device
#   - Alert check: every ~10 seconds per device

print("Understand the weight system, then implement in locust/locustfile.py")

# Locust test progression for your report:
test_plan = [
    {"users": 10,   "spawn_rate": 2,  "observe": "Baseline — all metrics should be green"},
    {"users": 100,  "spawn_rate": 10, "observe": "Light load — P95 should stay < 200ms"},
    {"users": 500,  "spawn_rate": 25, "observe": "Medium load — watch for latency increase"},
    {"users": 1000, "spawn_rate": 50, "observe": "High load — find the bottleneck"},
]
for step in test_plan:
    print(f"  {step['users']:5d} users: {step['observe']}")

---

## 📌 YOUR TASK — File Map

### Step 1 — Implement Test Fixtures (`tests/conftest.py`)

| Fixture | What to implement |
|---------|-------------------|
| `test_db()` | SQLite in-memory engine, create all tables, yield session, drop all after |
| `client(test_db)` | Override `get_db` dependency, yield `TestClient(app)`, clear overrides |
| `registered_device(client)` | POST to `/devices/register`, assert 201, return `device_id` |

### Step 2 — Device Tests (`tests/test_devices.py`)

Implement all 7 test methods. Required coverage:
- Successful registration → 201
- Duplicate registration → 409
- Missing required field → 422
- device_id too short → 422
- Optional firmware_version stored correctly
- Get existing device → 200
- Get nonexistent device → 404

### Step 3 — Telemetry Tests (`tests/test_telemetry.py`)

Implement all test methods covering the edge case matrix:
- Valid upload → 201
- `gps_lat` too high (95.0) → 422
- `gps_lat` too low (-91.0) → 422
- `gps_lon` too high (200.0) → 422
- `gps_lon` too low (-181.0) → 422
- Unknown device_id → 404
- Negative obstacle distance → 422
- battery_level > 100 → 422
- History of new device returns `[]` → 200
- Upload 3 records → history has 3 items

### Step 4 — Alert Tests (`tests/test_alerts.py`)

Implement all 5 test methods:
- No alerts → `[]` (not 404)
- Critical telemetry creates alert in history
- Filter by device_id returns only that device's alerts
- `limit` parameter respected
- NONE alert_type does NOT create an alert record

### Step 5 — Verify Coverage

```bash
pytest tests/ -v --cov=app --cov-report=term-missing
# Target: > 80% overall coverage
# Look at the 'Missing' column — those lines need more tests
```

### Step 6 — Implement Locust Load Test (`locust/locustfile.py`)

| Method | What to implement |
|--------|-------------------|
| `on_start()` | Generate unique `device_id`, POST `/devices/register` |
| `upload_telemetry()` | POST valid telemetry with random Berkeley GPS coords |
| `check_alert_history()` | GET `/alerts/history?device_id={self.device_id}&limit=5` |

### Step 7 — Run the Load Test

```bash
# Start API with Docker Compose
docker-compose up --build

# In a second terminal, run Locust:
locust -f locust/locustfile.py --host=http://localhost:8000

# Open http://localhost:8089
# Test at: 10 → 100 → 500 → 1000 users
# Export the charts as screenshots for your report
```

---

## ✅ Lesson 4 Self-Assessment Checklist

**Unit Tests:**
- [ ] `pytest tests/ -v` passes with 0 failures
- [ ] At least 10 test functions implemented (not just `pass`)
- [ ] Overall test coverage ≥ 80% (`--cov-report=term-missing` shows this)
- [ ] All edge cases in the telemetry matrix are covered
- [ ] Alert history edge case tested: NONE telemetry creates no alert

**Load Test:**
- [ ] `locust/locustfile.py` registers a unique device per virtual user
- [ ] Load test runs successfully at 100 concurrent users with 0% failure rate
- [ ] Load test report (screenshot or CSV) saved showing QPS and P95 latency
- [ ] Brief written analysis: at what user count does P95 latency exceed 500ms?
- [ ] Hypothesis about the bottleneck: Is it CPU, DB connections, or something else?

**Commit:**
- [ ] GitHub commit: `test(all): implement 10+ unit tests with 80%+ coverage`

---

**When all boxes are checked → you are ready for Lesson 5.**